# EDA - Motor Master

Exploratory analysis of the motor policies master file.

In [0]:
%pip install openpyxl seaborn

In [0]:
df_motor = spark.read.parquet(
    "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings/policies_motor_master.parquet"
)
display(df_motor.limit(10))

In [0]:
# Trim to a smaller dataframe of features currently being used plus target and do EDA on that
variables_motor = [
    "period_start", "period_end", "exposure", "driver_age", "years_licensed", "ncb_years",
    "vehicle_value_group", "vehicle_age", "make_model", "engine_size_cc", "body_type", "transmission",
    "seats", "fuel_type", "driver_assist", "cover_type", "annual_mileage", "overnight_parking",
    "vehicle_use", "named_young_driver", "telematics_opt_in", "telematics_score", "voluntary_excess",
    "compulsory_excess", "channel", "payment_method", "missed_payments_12m", "new_business",
    "renewal_count", "days_quote_to_inception", "written_premium", "hail_days", "occupation",
    "marital_status", "credit_band", "prior_claims_5y", "licence_points", "branch_id",
    "marketing_segment", "region", "urban_density", "ultimate_claims",
]

df_motor_trim = df_motor.select(
    "address_id",
    "customer_id",
    "row_id",
    "policy_id",
    "earned_premium",
    *variables_motor,
).toPandas()
display(df_motor_trim.head(10))

In [0]:
import pandas as pd

df_motor_trim["Year"] = pd.to_datetime(df_motor_trim["period_start"]).dt.year
variables_motor.append("Year")

In [0]:
categorical_motor = [
    c for c in variables_motor
    if c not in ("period_start", "period_end")
    and (df_motor_trim[c].dtype in ("object", "bool") or c == "Year")
]

counts = {c: df_motor_trim[c].value_counts(dropna=False) for c in categorical_motor}

for c, vc in counts.items():
    print(f"--- {c} ---")
    print(vc)

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Motor_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for c, vc in counts.items():
        vc.rename("count").rename_axis(c).to_excel(writer, sheet_name=c)

print(f"Saved value counts to {excel_path}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_motor = [
    c for c in variables_motor
    if c != "Year" and pd.api.types.is_numeric_dtype(df_motor_trim[c])
]

for c in numeric_motor:
    sns.boxplot(data=df_motor_trim, y=c)
    plt.title(c)
    plt.show()

In [0]:
import io
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Motor_EDA.xlsx"

numeric_motor = [
    c for c in variables_motor
    if c != "Year" and pd.api.types.is_numeric_dtype(df_motor_trim[c])
]

wb = load_workbook(excel_path)

for c in numeric_motor:
    fig, ax = plt.subplots()
    sns.boxplot(data=df_motor_trim, y=c, ax=ax)
    ax.set_title(c)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    buf.seek(0)
    img = XLImage(buf)
    ws = wb.create_sheet(f"{c}_boxplot"[:31])
    ws.add_image(img, "B2")

wb.save(excel_path)
print(f"Added {len(numeric_motor)} boxplots to {excel_path}")

In [0]:
over_one = df_motor_trim[df_motor_trim["exposure"] > 1]

print(f"Rows with exposure > 1: {len(over_one):,} of {len(df_motor_trim):,} ({len(over_one) / len(df_motor_trim):.2%})")
print(over_one["exposure"].describe())

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Motor_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    over_one.to_excel(writer, sheet_name="exposure_over_1", index=False)

print(f"Saved {len(over_one):,} rows to {excel_path} (sheet: exposure_over_1)")

3,763 of 793,414 rows (0.47%) have exposure > 1
Of those, the median is just 1.0139, but the mean is 63.9 and the max is 365.0 — so this is the same mixed-scale problem you found in the buildings data: a tail of rows storing day counts (365) instead of year fractions, plus a cluster just over 1 (1.0139 ≈ 365/360 — looks like a leap-year/divisor inconsistency)
Decision to fix exposure column